# 🧮 Taller computacional: el Teorema del Residuo Chino

**Estudiante:** Giovanny Antonio Carreño Barragán  
**Código:** 2262203  
**Matemáticas Discretas · Ciencia de Datos · 2do semestre**  
**Duración:** 2 horas · **Modalidad:** individual, cada estudiante en su computador  
**Referencia:** KH Rosen, *Matemáticas Discretas y sus Aplicaciones* (8.ª ed.), sección 4.4 *Resolución de Congruencias*.

---

### Objetivos de aprendizaje
Al terminar el taller podrás:
1. Implementar el **algoritmo extendido de Euclides** y usarlo para calcular **inversos modulares**.
2. Implementar el **Teorema del Residuo Chino (TRC)** siguiendo la construcción de la demostración de Rosen.
3. Extender el algoritmo a sistemas con **módulos que no son primos relativos** y detectar sistemas sin solución.
4. **Medir y comparar** empíricamente el costo de la fuerza bruta frente al TRC.
5. Usar la **representación por residuos** para hacer aritmética con enteros grandes, de forma vectorizada con NumPy, y evitar el desbordamiento de `int64`.

### Cómo trabajar
- Ejecuta las celdas **en orden** (`Shift + Enter`).
- Donde veas `# TU CÓDIGO AQUÍ` debes completar la función y borrar la línea `raise NotImplementedError`.
- Después de cada ejercicio hay una **celda de verificación**. Si imprime ✅, sigue adelante; si lanza un `AssertionError`, lee el mensaje y corrige.
- Las preguntas marcadas con ✍️ se responden en la celda de texto que les sigue.
- **No está permitido** usar funciones que ya resuelven el problema (`sympy.ntheory.modular.crt`, `pow(a, -1, m)` dentro de tus funciones, etc.). Sí puedes usarlas **para verificar**.

### Cronograma sugerido

| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación del entorno y repaso | 5 min |
| 1 | Fuerza bruta: el acertijo de Sun-Tsu | 10 min |
| 2 | Euclides extendido e inversos modulares | 20 min |
| 3 | El TRC constructivo | 20 min |
| 4 | Sistemas con módulos no coprimos | 20 min |
| 5 | Experimento de rendimiento | 15 min |
| 6 | Aritmética con enteros grandes (representación por residuos) | 25 min |
| 7 | Cierre y entrega | 5 min |
| ⭐ | Reto opcional: RSA acelerado con el TRC | extra |

## Parte 0 · Preparación (5 min)

Ejecuta la siguiente celda. Si alguna librería falta, instálala con `pip install numpy pandas matplotlib`.

In [ ]:
import math
import random
import time
from functools import reduce
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(2026)
print("Python listo ✔ | numpy", np.__version__, "| pandas", pd.__version__)

### Repaso rápido

**Teorema del Residuo Chino (Rosen, Teorema 2).** Sean $m_1, m_2, \dots, m_n$ enteros positivos mayores que 1, **primos relativos dos a dos**, y $a_1, \dots, a_n$ enteros arbitrarios. Entonces el sistema

$$x \equiv a_1 \pmod{m_1},\quad x \equiv a_2 \pmod{m_2},\quad \dots,\quad x \equiv a_n \pmod{m_n}$$

tiene **solución única módulo** $m = m_1 m_2 \cdots m_n$.

**Construcción de la demostración.** Sea $M_k = m / m_k$ y sea $y_k$ un inverso de $M_k$ módulo $m_k$ (existe porque $\gcd(M_k, m_k) = 1$). Entonces

$$x = a_1 M_1 y_1 + a_2 M_2 y_2 + \cdots + a_n M_n y_n$$

es solución, porque $M_j \equiv 0 \pmod{m_k}$ si $j \ne k$ y $M_k y_k \equiv 1 \pmod{m_k}$.

**El acertijo de Sun-Tsu (siglo I):** *Hay cierta cantidad de cosas. Al dividirlas entre 3 sobran 2; entre 5 sobran 3; entre 7 sobran 2. ¿Cuántas cosas hay?*

## Parte 1 · Fuerza bruta (10 min)

La forma más ingenua de resolver un sistema de congruencias es **probar todos los candidatos** $x = 0, 1, 2, \dots$ hasta encontrar uno que cumpla todas las congruencias. Si existe solución, aparece antes de llegar a $\operatorname{mcm}(m_1, \dots, m_n) \le m_1 m_2 \cdots m_n$.

### Ejercicio 1.1
Implementa `crt_fuerza_bruta(residuos, modulos)` que devuelva el **menor** $x \ge 0$ que satisface todas las congruencias, o `None` si no hay ninguno en el rango $0 \le x < m_1 m_2 \cdots m_n$.

> 💡 `math.prod(lista)` calcula el producto; `zip(residuos, modulos)` recorre las parejas $(a_k, m_k)$.

In [ ]:
def crt_fuerza_bruta(residuos, modulos):
    limite = math.prod(modulos)
    for x in range(limite):
        # revisamos si x cumple con cada una de las congruencias
        cumple = True
        for a, m in zip(residuos, modulos):
            if x % m != a % m:
                cumple = False
                break
        if cumple:
            return x
    return None

In [ ]:
# ✅ Verificación 1.1
assert crt_fuerza_bruta([2, 3, 2], [3, 5, 7]) == 23, "El acertijo de Sun-Tsu (Ejemplo 4) tiene solución 23"
assert crt_fuerza_bruta([1, 2, 3], [5, 6, 7]) == 206, "Ejemplo 6 de Rosen: x ≡ 206 (mod 210)"
assert crt_fuerza_bruta([1, 2], [4, 6]) is None, "x ≡ 1 (mod 4) y x ≡ 2 (mod 6) no tiene solución"
assert crt_fuerza_bruta([-1, 7], [3, 5]) == 2, "Los residuos negativos o mayores que el módulo también deben funcionar"
print("✅ Ejercicio 1.1 correcto. Sun-Tsu tenía", crt_fuerza_bruta([2, 3, 2], [3, 5, 7]), "cosas.")

### Pregunta 1.2 ✍️
En el **peor caso**, ¿cuántas veces se evalúa la condición `x % m == a % m` en función de los módulos? ¿Qué pasa con el tiempo de ejecución si agregas una congruencia más con un módulo de tamaño ~100?

**✍️ Tu respuesta:**

- En el peor de los casos, la condición se evalúa hasta $n \cdot M$ veces (donde $n$ es la cantidad de congruencias y $M = \prod m_k$ es el producto de los módulos). Esto ocurre cuando la solución es el último valor del rango ($x = M - 1$) o cuando el sistema no tiene solución y el bucle tiene que revisar todos los candidatos.
- Si añadimos otra congruencia con un módulo cercano a 100, el espacio de búsqueda se multiplica por 100. Por lo tanto, el programa tardaría aproximadamente **100 veces más** en terminar. El tiempo crece de manera exponencial respecto a la cantidad de ecuaciones, haciendo que la fuerza bruta sea inviable muy rápido.

## Parte 2 · Euclides extendido e inversos modulares (20 min)

El paso clave de la construcción es encontrar $y_k$ tal que $M_k y_k \equiv 1 \pmod{m_k}$. Para eso usamos el **algoritmo extendido de Euclides**: dados $a, b$, encuentra $g = \gcd(a, b)$ y enteros $s, t$ (coeficientes de Bézout) con

$$g = s\,a + t\,b.$$

Si $\gcd(a, m) = 1$, entonces $s\,a + t\,m = 1$, así que $s\,a \equiv 1 \pmod m$: **$s$ es un inverso de $a$ módulo $m$**.

### Ejercicio 2.1
Implementa `euclides_extendido(a, b)` de forma **iterativa** (sin recursión). Debe devolver la tupla `(g, s, t)`.

> 💡 Mantén dos filas $(r_0, s_0, t_0)$ y $(r_1, s_1, t_1)$ que siempre cumplen $r_i = s_i a + t_i b$. Empieza con $(a, 1, 0)$ y $(b, 0, 1)$. En cada paso calcula `q = r0 // r1` y reemplaza la fila vieja por `fila0 - q * fila1`.

In [ ]:
def euclides_extendido(a, b):
    # iniciamos con las dos filas base para calcular los coeficientes de Bezout
    r0, r1 = a, b
    s0, s1 = 1, 0
    t0, t1 = 0, 1
    
    while r1 != 0:
        q = r0 // r1
        r0, r1 = r1, r0 - q * r1
        s0, s1 = s1, s0 - q * s1
        t0, t1 = t1, t0 - q * t1
        
    return r0, s0, t0

In [ ]:
# ✅ Verificación 2.1
g, s, t = euclides_extendido(4620, 101)
assert g == 1 and s * 4620 + t * 101 == 1, f"Se esperaba 1 = s·4620 + t·101, obtuviste g={g}, s={s}, t={t}"
print(f"Ejemplo 2 de Rosen: 1 = ({s})·4620 + ({t})·101")

for _ in range(2000):
    a, b = random.randint(1, 10**12), random.randint(1, 10**12)
    g, s, t = euclides_extendido(a, b)
    assert g == math.gcd(a, b), f"gcd incorrecto para ({a}, {b})"
    assert s * a + t * b == g, f"Los coeficientes de Bézout no cumplen la identidad para ({a}, {b})"
print("✅ Ejercicio 2.1 correcto (2000 pruebas aleatorias con números de hasta 12 dígitos).")

### Ejercicio 2.2
Implementa `inverso_modular(a, m)` que devuelva el inverso de $a$ módulo $m$ **en el rango** $0 \le y < m$. Si $\gcd(a, m) \neq 1$ debe lanzar `ValueError`.

In [ ]:
def inverso_modular(a, m):
    g, s, _ = euclides_extendido(a, m)
    if g != 1:
        raise ValueError(f"{a} no tiene inverso modulo {m}")
    return s % m

In [ ]:
# ✅ Verificación 2.2
assert inverso_modular(101, 4620) == 1601, "Rosen: 1601 es inverso de 101 módulo 4620"
assert inverso_modular(3, 7) == 5, "−2 ≡ 5 es inverso de 3 módulo 7 (Ejemplo 1)"
try:
    inverso_modular(6, 9)
    raise AssertionError("inverso_modular(6, 9) debería lanzar ValueError")
except ValueError:
    pass
for _ in range(2000):
    m = random.randint(2, 10**9)
    a = random.randint(1, 10**12)
    if math.gcd(a, m) == 1:
        y = inverso_modular(a, m)
        assert 0 <= y < m and y == pow(a, -1, m), f"Inverso incorrecto para a={a}, m={m}"
print("✅ Ejercicio 2.2 correcto.")

### Ejercicio 2.3
Usa tu inverso para resolver la congruencia lineal $ax \equiv b \pmod m$ cuando $\gcd(a, m) = 1$. Devuelve la solución en $0 \le x < m$.

In [ ]:
def resolver_congruencia_lineal(a, b, m):
    # multiplicamos b por el inverso modular de a
    inv_a = inverso_modular(a, m)
    return (inv_a * b) % m

In [ ]:
# ✅ Verificación 2.3
assert resolver_congruencia_lineal(3, 4, 7) == 6, "Ejemplo 3 de Rosen: 3x ≡ 4 (mod 7) ⇒ x ≡ 6"
assert resolver_congruencia_lineal(5, 1, 6) == 5, "Ejemplo 6 de Rosen: 5t ≡ 1 (mod 6) ⇒ t ≡ 5"
assert resolver_congruencia_lineal(30, -23, 7) == 6, "Ejemplo 6 de Rosen: 30u ≡ 3 − 26 (mod 7) ⇒ u ≡ 6"
print("✅ Ejercicio 2.3 correcto.")

### Ejercicio 2.4 · ¿Qué tan rápido es Euclides?
Implementa `pasos_euclides(a, b)` que cuente cuantas divisiones hace el algoritmo de Euclides. Luego ejecuta el experimento: compara pares aleatorios con pares de **números de Fibonacci consecutivos** del mismo tamaño.

In [ ]:
def pasos_euclides(a, b):
    divisiones = 0
    while b != 0:
        a, b = b, a % b
        divisiones += 1
    return divisiones

In [ ]:
# ✅ Verificación 2.4 + experimento (el código del experimento ya está escrito)
assert pasos_euclides(4620, 101) == 7, "gcd(4620, 101) requiere 7 divisiones (la última con residuo 0)"

fib = [1, 1]
while len(fib) < 200:
    fib.append(fib[-1] + fib[-2])

digitos, pasos_fib, pasos_aleat = [], [], []
for k in range(5, 200, 5):
    a, b = fib[k + 1], fib[k]
    d = len(str(a))
    muestras = [pasos_euclides(random.randint(10**(d-1), 10**d), random.randint(10**(d-1), 10**d)) for _ in range(200)]
    digitos.append(d)
    pasos_fib.append(pasos_euclides(a, b))
    pasos_aleat.append(np.mean(muestras))

plt.figure(figsize=(7, 4))
plt.plot(digitos, pasos_fib, "o-", label="Fibonacci consecutivos")
plt.plot(digitos, pasos_aleat, "s-", label="Pares aleatorios (promedio de 200)")
plt.xlabel("Número de dígitos decimales")
plt.ylabel("Divisiones realizadas")
plt.title("Costo del algoritmo de Euclides")
plt.legend()
plt.grid(alpha=.3)
plt.show()
print("✅ Ejercicio 2.4 correcto.")

### Pregunta 2.5 ✍️
¿Cómo crece el número de pasos con el número de dígitos: lineal, cuadrático o exponencial? ¿Qué pares son el peor caso? ¿Por qué esto hace que el algoritmo sea práctico incluso con números de cientos de dígitos?

**✍️ Tu respuesta:**

- El número de pasos crece de forma **lineal** con respecto a la cantidad de dígitos decimales (lo cual equivale a un crecimiento logarítmico con respecto al valor del número en sí).
- El **peor caso** ocurre cuando calculamos el mcd de dos términos consecutivos de la sucesión de Fibonacci ($F_{k+1}, F_k$). En cada división, el cociente siempre es 1 y el residuo es el término anterior de Fibonacci, lo que hace que los números disminuyan lo más lento posible.
- Esto hace que sea muy práctico porque, por el **Teorema de Lamé**, la cantidad de pasos nunca supera 5 veces la cantidad de dígitos decimales del número menor. Así tengamos números de 300 dígitos (como en criptografía RSA), el algoritmo hace a lo sumo unas 1500 divisiones, algo que la computadora hace en fracciones de milisegundo.

## Parte 3 · El TRC constructivo (20 min)

### Ejercicio 3.1
Implementa `son_coprimos_dos_a_dos(modulos)` que devuelva `True` si $\gcd(m_i, m_j) = 1$ para **todo** par $i \ne j$.

> 💡 `itertools.combinations(modulos, 2)` genera todos los pares.

In [ ]:
def son_coprimos_dos_a_dos(modulos):
    for m1, m2 in combinations(modulos, 2):
        if math.gcd(m1, m2) != 1:
            return False
    return True

In [ ]:
# ✅ Verificación 3.1
assert son_coprimos_dos_a_dos([3, 5, 7])
assert son_coprimos_dos_a_dos([99, 98, 97, 95]), "Los módulos del Ejemplo 8 son coprimos dos a dos"
assert not son_coprimos_dos_a_dos([4, 9, 6]), "4 y 6 comparten el factor 2"
assert son_coprimos_dos_a_dos([6, 35, 143]), "Ojo: 6·35·143 no son primos, pero sí son coprimos dos a dos"
print("✅ Ejercicio 3.1 correcto.")

### Ejercicio 3.2 · El corazón del taller
Implementa `crt(residuos, modulos)` siguiendo **exactamente** la construcción de la demostración de Rosen:

1. Si los módulos no son coprimos dos a dos, lanza `ValueError`.
2. Calcular $m = \prod m_k$.
3. Para cada $k$: $M_k = m // m_k$, $y_k = $ inverso de $M_k$ módulo $m_k$, y acumula $a_k M_k y_k$.
4. Devuelve la suma reducida módulo $m$.

In [ ]:
def crt(residuos, modulos):
    if not son_coprimos_dos_a_dos(modulos):
        raise ValueError("Los módulos no son coprimos dos a dos")
    
    m = math.prod(modulos)
    suma = 0
    for a_k, m_k in zip(residuos, modulos):
        M_k = m // m_k
        y_k = inverso_modular(M_k, m_k)
        suma += a_k * M_k * y_k
        
    return suma % m

In [ ]:
# ✅ Verificación 3.2
assert crt([2, 3, 2], [3, 5, 7]) == 23, "Ejemplo 5 de Rosen"
assert crt([1, 2, 3], [5, 6, 7]) == 206, "Ejemplo 6 de Rosen"
assert crt([65, 2, 51, 10], [99, 98, 97, 95]) == 537140, "Ejemplo 8 de Rosen"
try:
    crt([1, 2], [4, 6])
    raise AssertionError("crt debe lanzar ValueError si los módulos no son coprimos")
except ValueError:
    pass

# Comparación contra fuerza bruta en sistemas pequeños
primos_peq = [2, 3, 5, 7, 11, 13, 17]
for _ in range(300):
    mods = random.sample(primos_peq, 3)
    res = [random.randint(-50, 50) for _ in mods]
    assert crt(res, mods) == crt_fuerza_bruta(res, mods), f"Falla con residuos {res} y módulos {mods}"

# Prueba con números grandes: escogemos x, calculamos residuos y debemos recuperar x
mods = [2**61 - 1, 2**89 - 1, 2**107 - 1, 10**18 + 9]
for _ in range(300):
    x = random.randrange(math.prod(mods))
    assert crt([x % m for m in mods], mods) == x, "No se recuperó el número original"
print("✅ Ejercicio 3.2 correcto. Tu TRC funciona con números de", len(str(math.prod(mods))), "dígitos.")

### Ejercicio 3.3 · Traza del algoritmo como tabla
En ciencia de datos es útil **inspeccionar** los pasos intermedios. Implementa `crt_traza(residuos, modulos)` que devuelva un `DataFrame` de pandas con una fila por congruencia y las columnas `m_k`, `a_k`, `M_k`, `y_k`, `a_k·M_k·y_k`. La suma del último módulo $m$ debe ser la solución.

In [ ]:
def crt_traza(residuos, modulos):
    m = math.prod(modulos)
    filas = []
    for a_k, m_k in zip(residuos, modulos):
        M_k = m // m_k
        y_k = inverso_modular(M_k, m_k)
        termino = a_k * M_k * y_k
        filas.append({
            "m_k": m_k,
            "a_k": a_k,
            "M_k": M_k,
            "y_k": y_k,
            "a_k·M_k·y_k": termino
        })
    return pd.DataFrame(filas)

In [ ]:
# ✅ Verificación 3.3
traza = crt_traza([2, 3, 2], [3, 5, 7])
display(traza)
total = traza["a_k·M_k·y_k"].sum()
print(f"Suma = {total} ≡ {total % 105} (mod 105)")
assert list(traza["M_k"]) == [35, 21, 15] and list(traza["y_k"]) == [2, 1, 1], "Compara con el Ejemplo 5 de Rosen"
assert total == 233
print("✅ Ejercicio 3.3 correcto: coincide con el Ejemplo 5 (233 ≡ 23 mod 105).")

## Parte 4 · Sistemas con módulos no coprimos (20 min)

El TRC exige módulos coprimos, pero en la práctica (horarios, ciclos, calendarios) aparecen sistemas como $x \equiv 2 \pmod 4$, $x \equiv 4 \pmod 6$. La idea del **método de sustitución hacia atrás** (Ejemplo 6 de Rosen) generaliza bien: combinamos las congruencias **de dos en dos**.

Para combinar $x \equiv a_1 \pmod{m_1}$ y $x \equiv a_2 \pmod{m_2}$:

1. Escribimos $x = a_1 + m_1 k$. Sustituyendo: $m_1 k \equiv a_2 - a_1 \pmod{m_2}$.
2. Sea $g = \gcd(m_1, m_2)$. **Hay solución si y solo si** $g \mid (a_2 - a_1)$.
3. Si $m_1 s + m_2 t = g$ (Euclides extendida), entonces $k \equiv \dfrac{a_2 - a_1}{g}\, s \pmod{\dfrac{m_2}{g}}$.
4. La solución combinada es $x \equiv a_1 + m_1 k \pmod{\operatorname{mcm}(m_1, m_2)}$, con $\operatorname{mcm}(m_1, m_2) = \dfrac{m_1 m_2}{g}$.

### Ejercicio 4.1
Implementa `combinar(a1, m1, a2, m2)` que devuelva la tupla `(a, mcm)` con $0 \le a < \operatorname{mcm}$, o `None` si el par es inconsistente.

In [ ]:
def combinar(a1, m1, a2, m2):
    g, s, _ = euclides_extendido(m1, m2)
    diff = a2 - a1
    # Si el mcd no divide a la diferencia, no hay solucion
    if diff % g != 0:
        return None
    
    mcm = (m1 * m2) // g
    mod_k = m2 // g
    k = ((diff // g) * s) % mod_k
    x = (a1 + m1 * k) % mcm
    return x, mcm

In [ ]:
# ✅ Verificación 4.1
assert combinar(2, 4, 4, 6) == (10, 12), "x ≡ 2 (mod 4), x ≡ 4 (mod 6) ⇒ x ≡ 10 (mod 12)"
assert combinar(1, 4, 2, 6) is None, "x ≡ 1 (mod 4), x ≡ 2 (mod 6) es inconsistente (paridad)"
assert combinar(1, 5, 2, 6) == (26, 30), "Primer paso del Ejemplo 6: x = 30u + 26"
assert combinar(26, 30, 3, 7) == (206, 210), "Segundo paso del Ejemplo 6"
print("✅ Ejercicio 4.1 correcto.")

### Ejercicio 4.2
Implementa `crt_general(residuos, modulos)` que combine todas las congruencias una por una y devuelva `(x, L)` donde $L$ es el mcm de los módulos y $0 \le x < L$, o `None` si el sistema no tiene solución.

> 💡 Empieza con la congruencia trivial $x \equiv 0 \pmod 1$ y ve combinando. (Si te sientes cómodo, prueba con `functools.reduce`).

In [ ]:
def crt_general(residuos, modulos):
    # empezamos con la solucion base x = 0 (mod 1)
    x, m = 0, 1
    for a_i, m_i in zip(residuos, modulos):
        resultado = combinar(x, m, a_i, m_i)
        if resultado is None:
            return None
        x, m = resultado
    return x, m

In [ ]:
# ✅ Verificación 4.2
assert crt_general([1, 2, 3], [5, 6, 7]) == (206, 210), "Ejemplo 6 de Rosen"
assert crt_general([2, 3, 2], [3, 5, 7]) == (23, 105), "Sun-Tsu"
assert crt_general([2, 4, 5], [4, 6, 9]) is None, "x ≡ 4 (mod 6) exige x ≡ 1 (mod 3), pero x ≡ 5 (mod 9) exige x ≡ 2 (mod 3)"
assert crt_general([3, 3, 9], [4, 6, 15]) == (39, 60)

# Prueba masiva contra fuerza bruta, con módulos que comparten factores
inconsistentes = 0
for _ in range(3000):
    mods = [random.randint(2, 12) for _ in range(3)]
    res = [random.randint(0, 30) for _ in range(3)]
    esperado = crt_fuerza_bruta(res, mods)
    obtenido = crt_general(res, mods)
    if esperado is None:
        inconsistentes += 1
        assert obtenido is None, f"{res}, {mods} no tiene solución pero devolviste {obtenido}"
    else:
        assert obtenido is not None and obtenido[0] == esperado, f"Falla con {res}, {mods}"
        assert obtenido[1] == math.lcm(*mods)
print(f"✅ Ejercicio 4.2 correcto. De 3000 sistemas aleatorios, {inconsistentes} no tenían solución "
      f"({inconsistentes/30:.1f} %). cuadros")

### Pregunta 4.3 ✍️
En la prueba masiva anterior, ¿aproximadamente qué porcentaje de sistemas aleatorios **no** tenía solución? ¿Por qué tantos? ¿Qué condición garantiza que un sistema *siempre* tenga solución, sin importar los residuos?

**✍️ Tu respuesta:**

- Alrededor del **50% al 55%** de los sistemas aleatorios resultaron no tener solución.
- Salen tantos sin solución porque los enteros pequeños entre 2 y 12 suelen compartir factores primos con mucha frecuencia (por ejemplo, números pares que comparten el 2, o múltiplos de 3 como 3, 6, 9, 12). Si dos módulos $m_1$ y $m_2$ tienen $\gcd(m_1, m_2) = g > 1$, se necesita obligatoriamente que $a_1 \equiv a_2 \pmod g$. Al generar los residuos al azar, es muy poco probable que coincidan en ese módulo compartido; basta con que un solo par falle para que todo el sistema sea inconsistente.
- La condición que garantiza que siempre exista una solución única (sin importar qué residuos tengamos) es que los módulos sean **primos relativos dos a dos** (es decir, $\gcd(m_i, m_j) = 1$ para todo $i \ne j$), que es precisamente la condición del Teorema del Residuo Chino clásico.

## Parte 5 · Experimento de rendimiento (15 min)

Vamos a medir empíricamente cuánto tarda cada método cuando agregamos más congruencias. Usaremos como módulos los primeros $k$ primos y el **peor caso** para la fuerza bruta: residuos $a_k = m_k - 1$ (la solución es $x = m - 1$, el último candidato).

### Ejercicio 5.1
Completa la función `medir(funcion, residuos, modulos, repeticiones)` que devuelva el **mínimo** tiempo (en segundos) de varias ejecuciones. Usa `time.perf_counter()`.

> 💡 Se usa el mínimo, no el promedio, porque el ruido del sistema operativo solo puede *sumar* tiempo.

In [ ]:
def medir(funcion, residuos, modulos, repeticiones=3):
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion(residuos, modulos)
        fin = time.perf_counter()
        tiempos.append(fin - inicio)
    return min(tiempos)

In [ ]:
# ✅ Experimento 5.1 (puede tardar unos 10–20 segundos)
def primeros_primos(n):
    primos, candidato = [], 2
    while len(primos) < n:
        if all(candidato % p for p in primos if p * p <= candidato):
            primos.append(candidato)
        candidato += 1
    return primos

PRIMOS = primeros_primos(200)
filas = []
for k in range(1, 201):
    mods = PRIMOS[:k]
    res = [m - 1 for m in mods]
    fila = {"k": k, "digitos_m": len(str(math.prod(mods))), "t_crt": medir(crt, res, mods)}
    fila["t_fuerza_bruta"] = medir(crt_fuerza_bruta, res, mods, 1) if k <= 7 else np.nan
    filas.append(fila)

resultados = pd.DataFrame(filas)
assert resultados["t_crt"].gt(0).all(), "medir debe devolver tiempos positivos"
display(resultados.head(10))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(resultados["k"], resultados["t_fuerza_bruta"], "o-", label="Fuerza bruta")
ax.semilogy(resultados["k"], resultados["t_crt"], ".-", label="TRC constructivo")
ax.set_xlabel("k (número de congruencias; módulos = primeros k primos)")
ax.set_ylabel("Tiempo (s, escala log)")
ax.set_title("Fuerza bruta vs. TRC"); ax.legend(); ax.grid(alpha=.3, which="both")
plt.show()
print("✅ Ejercicio 5.1 completado.")

### Ejercicio 5.2 · Extrapolación
Con los datos de la fuerza bruta, estima el **tiempo por candidato** (segundos por cada $x$ probado) y úsalo para predecir cuánto tardaría la fuerza bruta con los **primeros 20 primos** y con los **primeros 200 primos**. Expresa el resultado en años y compáralo con lo que tardó tu `crt` con $k = 200$.

> 💡 Un año tiene unos $3.15 \times 10^7$ segundos. La edad del universo es ~$1.4 \times 10^{10}$ años.
> ⚠️ El producto de los primeros 200 primos tiene más de 500 dígitos y **no cabe en un `float`** (máximo ≈ $1.8 \times 10^{308}$). Para ese caso trabaja con logaritmos: `math.log10` acepta enteros de Python de cualquier tamaño.

In [ ]:
# Estimamos el tiempo por candidato usando k = 7 (el caso mas grande medido con fuerza bruta)
m_7 = math.prod(PRIMOS[:7])
t_7 = resultados.loc[resultados["k"] == 7, "t_fuerza_bruta"].iloc[0]
tiempo_por_candidato = t_7 / m_7

segundos_por_anio = 3.15e7

# Estimacion para 20 primos
m_20 = math.prod(PRIMOS[:20])
anios_20 = (m_20 * tiempo_por_candidato) / segundos_por_anio

# Para 200 primos usamos log10 para evitar desbordamiento del tipo float
m_200 = math.prod(PRIMOS[:200])
log10_anios_200 = math.log10(m_200) + math.log10(tiempo_por_candidato) - math.log10(segundos_por_anio)

print(f"Tiempo por candidato ≈ {tiempo_por_candidato:.2e} s")
print(f"Fuerza bruta con 20 primos ≈ {anios_20:.2e} años")
print(f"Fuerza bruta con 200 primos ≈ 10^{log10_anios_200:.1f} años")
print(f"TRC con 200 primos: {resultados['t_crt'].iloc[-1]*1000:.2f} ms")

### Pregunta 5.3 ✍️
En la gráfica, ¿por qué la curva de la fuerza bruta se ve como una línea casi recta en escala logarítmica? ¿Cuál es el factor que domina el costo del TRC constructivo cuando $k$ crece?

**✍️ Tu respuesta:**

- La curva de la fuerza bruta se ve como una línea recta porque el eje vertical está en escala logarítmica. Al agregar cada nuevo primo, el número de candidatos a probar se multiplica por ese nuevo primo ($T_{k+1} \approx p_{k+1} \cdot T_k$), lo que representa un crecimiento exponencial respecto a $k$. Como el logaritmo de una función exponencial es lineal, en la gráfica se transforma en una recta.
- En el TRC constructivo la cantidad de pasos crece de forma polinomial. Sin embargo, cuando $k$ llega a 200, el producto $m$ supera los 500 dígitos decimales, por lo que el factor que domina el costo es la propia **aritmética de enteros gigantes (bignums)** en Python (las multiplicaciones y divisiones largas con números de cientos de dígitos).

## Parte 6 · Aritmética con enteros grandes (25 min)

Rosen (sección 4.4.4) explica que, si $m_1, \dots, m_n$ son coprimos dos a dos, todo entero $0 \le a < m$ se representa **de forma única** por la tupla

$$a \;\longleftrightarrow\; (a \bmod m_1,\; a \bmod m_2,\; \dots,\; a \bmod m_n).$$

Sumar y multiplicar se hace **componente a componente**, cada canal con números pequeños e **independiente de los demás** (¡paralelizable!). Solo al final se usa el TRC para reconstruir el resultado.

### Ejercicio 6.1 · La biyección del Ejemplo 7
Construye un `DataFrame` llamado `tabla` con las columnas `x`, `x mod 3`, `x mod 4` para $x = 0, \dots, 11$. Luego cuenta cuántas **parejas distintas** aparecen. Repite con los módulos 4 y 6 (que **no** son coprimos) para $x = 0, \dots, 23$ y guarda el conteo en `distintas_4_6`.

In [ ]:
tabla = pd.DataFrame({
    "x": list(range(12)),
    "x mod 3": [x % 3 for x in range(12)],
    "x mod 4": [x % 4 for x in range(12)]
})
distintas_3_4 = len(set(zip(tabla["x mod 3"], tabla["x mod 4"])))

parejas_4_6 = set((x % 4, x % 6) for x in range(24))
distintas_4_6 = len(parejas_4_6)

In [ ]:
# ✅ Verificación 6.1
display(tabla.T)
assert distintas_3_4 == 12, "Con módulos 3 y 4 las 12 parejas deben ser distintas (biyección)"
assert distintas_4_6 == 12, "Con 4 y 6 solo hay mcm(4,6) = 12 parejas distintas entre 24 números"
print(f"Módulos 3 y 4: {distintas_3_4} parejas distintas para 12 números → biyección ✔")
print(f"Módulos 4 y 6: {distintas_4_6} parejas distintas para 24 números → hay colisiones ✘")
print("✅ Ejercicio 6.1 correcto.")

### Ejercicio 6.2 · Un mini sistema de numeración por residuos
Implementa:
- `a_residuos(x, modulos)` → tupla de residuos.
- `desde_residuos(r, modulos)` → entero reconstruido (usa tu `crt`).
- `sumar_rns(r1, r2, modulos)` y `multiplicar_rns(r1, r2, modulos)` → operación componente a componente, reduciendo cada componente en su módulo.

In [ ]:
def a_residuos(x, modulos):
    return tuple(x % m for m in modulos)

def desde_residuos(r, modulos):
    return crt(r, modulos)

def sumar_rns(r1, r2, modulos):
    return tuple((a + b) % m for a, b, m in zip(r1, r2, modulos))

def multiplicar_rns(r1, r2, modulos):
    return tuple((a * b) % m for a, b, m in zip(r1, r2, modulos))

In [ ]:
# ✅ Verificación 6.2 — Ejemplo 8 de Rosen
MODS_8 = [99, 98, 97, 95]
ra, rb = a_residuos(123684, MODS_8), a_residuos(413456, MODS_8)
assert ra == (33, 8, 9, 89) and rb == (32, 92, 42, 16), "Compara con el Ejemplo 8"
rs = sumar_rns(ra, rb, MODS_8)
assert rs == (65, 2, 51, 10)
assert desde_residuos(rs, MODS_8) == 537140
print(f"{ra} + {rb} = {rs} → {desde_residuos(rs, MODS_8):,}")

M8 = math.prod(MODS_8)
for _ in range(1000):
    a, b = random.randrange(9000), random.randrange(9000) # a·b < 8.1e7 < M: cabe en el rango
    assert desde_residuos(multiplicar_rns(a_residuos(a, MODS_8), a_residuos(b, MODS_8), MODS_8), MODS_8) == a * b
print(f"✅ Ejercicio 6.2 correcto. Con estos módulos se representan enteros hasta {M8:,}.")

### Ejercicio 6.3 · Cuando NumPy se desborda
En ciencia de datos usamos arreglos de NumPy con tipos de **tamaño fijo** como `int64`. Si un resultado supera $2^{63} - 1 \approx 9.2 \times 10^{18}$, NumPy **se desborda en silencio**. Observa:

In [ ]:
valores = np.arange(1, 31, dtype=np.int64)
print("np.prod(1..30) =", np.prod(valores))
print("math.factorial(30) =", math.factorial(30))
print("¿Iguales?", int(np.prod(valores)) == math.factorial(30))

Vamos a calcular $30!$ **usando solo `int64`** con la representación por residuos. Rosen sugiere módulos de la forma $2^k - 1$, porque $\gcd(2^a - 1, 2^b - 1) = 2^{\gcd(a,b)} - 1$: si los exponentes son coprimos, los módulos también lo son.

Con exponentes $\le 31$, cada residuo es $< 2^{31}$, así que el producto de **dos** residuos es $< 2^{62}$ y **cabe en `int64`**.

Primero verifica computacionalmente esas dos afirmaciones:

In [ ]:
EXPONENTES = [31, 29, 23, 19, 17, 13, 11, 7]
MODS_RNS = [2**k - 1 for k in EXPONENTES]

# Verificación empírica de mcd(2^a − 1, 2^b − 1) = 2^mcd(a,b) − 1
assert all(math.gcd(2**a - 1, 2**b - 1) == 2**math.gcd(a, b) - 1 for a in range(1, 80) for b in range(1, 80))
print("Identidad mcd(2^a−1, 2^b−1) = 2^gcd(a,b) − 1 verificada para 1 ≤ a, b < 80")
print("¿Módulos coprimos dos a dos?", son_coprimos_dos_a_dos(MODS_RNS))
print(f"Rango representable: hasta ≈ 2^{math.log2(math.prod(MODS_RNS)):.0f} (30! ≈ 2^{math.log2(math.factorial(30)):.0f})")

Ahora implementa `producto_rns_numpy(valores, modulos)`:

1. Convierte `modulos` en un arreglo `np.int64` de forma `(n,)`.
2. Calcula la **matriz de residuos** `R = valores[:, None] % mods[None, :]` (cada fila es un número, cada columna un canal). Esto se llama *broadcasting*.
3. Recorre las filas acumulando `acum = (acum * fila) % mods`. Observa que cada paso opera **sobre todos los canales a la vez**.
4. Devuelve el entero reconstruido con `desde_residuos` (convierte a `int` de Python con `[int(r) for r in acum]`).

In [ ]:
def producto_rns_numpy(valores, modulos):
    mods = np.array(modulos, dtype=np.int64)
    vals = np.array(valores, dtype=np.int64)
    
    # matriz de residuos usando broadcasting
    R = vals[:, None] % mods[None, :]
    
    acum = np.ones(len(modulos), dtype=np.int64)
    for fila in R:
        acum = (acum * fila) % mods
        
    return desde_residuos([int(r) for r in acum], modulos)

In [ ]:
# ✅ Verificación 6.3
resultado = producto_rns_numpy(np.arange(1, 31, dtype=np.int64), MODS_RNS)
print("30! vía residuos + NumPy =", resultado)
assert resultado == math.factorial(30), "El producto reconstruido no coincide con 30!"

datos = np.random.default_rng(7).integers(10**6, 10**7, size=6, dtype=np.int64)
assert producto_rns_numpy(datos, MODS_RNS) == math.prod(int(v) for v in datos), "Falla con valores aleatorios"
print("✅ Ejercicio 6.3 correcto: calculaste un número de", len(str(resultado)),
      "dígitos usando solo aritmética int64.")

### Pregunta 6.4 ✍️
(a) ¿Qué pasaría si intentaras calcular $40!$ con estos mismos módulos? Compruébalo y explica.
(b) ¿Por qué decimos que este método es **paralelizable**? Relaciónalo con la forma de la matriz `R`.

**✍️ Tu respuesta:**

- **(a)** Si intentamos calcular $40!$ con estos módulos obtendríamos un resultado incorrecto. El rango máximo representable por este sistema es el producto $M = \prod (2^k - 1) \approx 2^{150} \approx 1.4 \times 10^{45}$, mientras que $40! \approx 8.16 \times 10^{47} \approx 2^{159.2}$. Como $40! > M$, el número se sale de la capacidad del sistema y el TRC nos devolvería el residuo $40! \bmod M$. Para solucionarlo tendríamos que incluir más módulos coprimos (como $2^{37}-1$) hasta que el producto total sea mayor que $40!$.
- **(b)** Es **paralelizable** porque cada columna de la matriz $R$ corresponde a un módulo independiente. Cuando sumamos o multiplicamos en el canal $k$, las operaciones solo dependen de los valores de esa columna modular y no hay acarreos ("carry") hacia las otras columnas. Por esto, cada canal se puede procesar al mismo tiempo en hilos diferentes o núcleos distintos sin necesidad de comunicarse entre sí hasta la reconstrucción final con el TRC.

## Parte 7 · Cierre y entrega (5 min)

Ejecuta la siguiente celda para ver un resumen de tu avance.

In [ ]:
pruebas = {
    "1.1 Fuerza bruta": lambda: crt_fuerza_bruta([2, 3, 2], [3, 5, 7]) == 23,
    "2.1 Euclides extendido": lambda: euclides_extendido(240, 46)[0] == 2,
    "2.2 Inverso modular": lambda: inverso_modular(101, 4620) == 1601,
    "2.3 Congruencia lineal": lambda: resolver_congruencia_lineal(3, 4, 7) == 6,
    "2.4 Pasos de Euclides": lambda: pasos_euclides(4620, 101) == 7,
    "3.1 Coprimos dos a dos": lambda: son_coprimos_dos_a_dos([99, 98, 97, 95]),
    "3.2 TRC constructivo": lambda: crt([65, 2, 51, 10], [99, 98, 97, 95]) == 537140,
    "3.3 Traza": lambda: crt_traza([2, 3, 2], [3, 5, 7])["a_k·M_k·y_k"].sum() == 233,
    "4.1 Combinar": lambda: combinar(2, 4, 4, 6) == (10, 12),
    "4.2 TRC general": lambda: crt_general([1, 2, 3], [5, 6, 7]) == (206, 210),
    "6.2 Aritmética por residuos": lambda: desde_residuos(sumar_rns((33, 8, 9, 89), (32, 92, 42, 16), [99, 98, 97, 95]), [99, 98, 97, 95]) == 537140,
    "6.3 Producto con NumPy": lambda: producto_rns_numpy(np.arange(1, 31), MODS_RNS) == math.factorial(30),
}
ok = 0
for nombre, prueba in pruebas.items():
    try:
        estado = "✅" if prueba() else "❌"
    except Exception as e:
        estado = f"⚠️ {type(e).__name__}"
    ok += estado == "✅"
    print(f"{estado} {nombre}")
print(f"\n{ok}/{len(pruebas)} ejercicios de código completos.")

### Reflexión final ✍️
En 3–5 líneas: ¿qué ventaja **computacional** concreta ofrece el TRC frente a resolver un sistema de congruencias "a mano" o por fuerza bruta? Menciona al menos un contexto de ciencia de datos o computación donde podría aparecer.

**✍️ Tu respuesta:**

El TRC convierte una búsqueda por fuerza bruta inviable de orden $\mathcal{O}(\prod m_k)$ en un cálculo directo y eficiente mediante el algoritmo extendido de Euclides. Además, la representación por residuos (RNS) permite descomponer cuentas con números gigantescos en varios canales modulares pequeños e independientes que se pueden procesar en paralelo sin acarreos entre sí, evitando desbordamientos de tipos fijos como `int64`. Esto se utiliza mucho en **criptografía** (para descifrar RSA mucho más rápido), en **computación homomórfica** sobre datos cifrados y en la multiplicación rápida de polinomios (NTT) aplicada en deep learning.

### Entrega
1. `Kernel → Restart & Run All` (o *Reiniciar y ejecutar todo*) y verifica que no haya errores.
2. Guarda el notebook como `taller_TRC_<tu_apellido>.ipynb` y súbelo a la plataforma del curso.

---
## ⭐ Reto opcional · RSA acelerado con el TRC

En RSA se descifra calculando $c^d \bmod n$ con $n = pq$. Como quien descifra conoce $p$ y $q$, puede calcular **dos potencias más pequeñas** y combinarlas con el TRC:

1. Por el **Pequeño Teorema de Fermat** (Rosen, Teorema 3), $c^{d} \equiv c^{d \bmod (p-1)} \pmod p$ (si $p \nmid c$). Sean $d_p = d \bmod (p-1)$ y $d_q = d \bmod (q-1)$.
2. $m_p = c^{d_p} \bmod p$, $\quad m_q = c^{d_q} \bmod q$.
3. $m = \text{TRC}([m_p, m_q], [p, q])$.

**Calentamiento:** verifica el Ejemplo 9 de Rosen, $7^{222} \bmod 11 = 5$, reduciendo el exponente con Fermat.

In [ ]:
exponente_reducido = 222 % (11 - 1)
print(f"7^222 ≡ 7^{exponente_reducido} = {7**exponente_reducido} ≡ {7**exponente_reducido % 11} (mod 11)")
assert pow(7, 222, 11) == 7**exponente_reducido % 11 == 5

Implementa `descifrar_rsa_crt(c, d, p, q)` y compara su tiempo con `pow(c, d, p*q)`. Usamos los primos de Mersenne $p = 2^{521}-1$ y $q = 2^{607}-1$ (puedes usar `pow(base, exp, mod)` de Python para las potencias modulares).

In [ ]:
def descifrar_rsa_crt(c, d, p, q):
    # reducimos los exponentes con el pequenio teorema de Fermat
    dp = d % (p - 1)
    dq = d % (q - 1)
    mp = pow(c, dp, p)
    mq = pow(c, dq, q)
    return crt([mp, mq], [p, q])

In [ ]:
# ✅ Verificación del reto
p, q = 2**521 - 1, 2**607 - 1
n, phi, e = p * q, (p - 1) * (q - 1), 65537
assert math.gcd(e, phi) == 1
d = inverso_modular(e, phi)

mensaje = random.randrange(2, n)
c = pow(mensaje, e, n)
assert descifrar_rsa_crt(c, d, p, q) == mensaje, "El mensaje descifrado no coincide"

def cronometrar(f, repeticiones=5):
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        f()
        tiempos.append(time.perf_counter() - inicio)
    return min(tiempos)

t_directo = cronometrar(lambda: pow(c, d, n))
t_crt = cronometrar(lambda: descifrar_rsa_crt(c, d, p, q))
print(f"Directo: {t_directo*1000:.2f} ms | Con TRC: {t_crt*1000:.2f} ms | Aceleración ≈ {t_directo/t_crt:.1f}×")
print("⭐ ¡Reto completado!")